# 3 Plotting with Seaborn

Seaborn is a data visualisation library for Python which builds on the
matplotlib package.

It is designed primarily with data exploration in mind. In particular:

- Seaborn integrates much more closely with pandas data structures
- It is capable of performing operations on entire datasets
- Its visualisation functions are designed to quickly produce detailed and
  informative statistical plots with few lines of code.

When importing seaborn, the convention is to use the alias `sns`:


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

We'll use a new dataset, `electricity_indicators` - the CCC's "key indicators"
for the electricity supply, taken from sheet `7.5.4` of the report. It holds a
set of metrics (operational capacity, levelised costs, gas share, and so on)
each recorded per year, with a `series` column splitting `Historical` values
from the projected `CCC milestone`:


In [ ]:
import jrpyvis as jrpy

indicators = jrpy.data.load("electricity_indicators")
indicators.head()

## 3.1 Seaborn as a wrapper

Let's motivate why seaborn is a good choice for statistical visualisations. To
do this, we'll create a regression plot (a scatter with a fitted trend line)
with seaborn, and then attempt to replicate it with matplotlib.

We'll look at one indicator - the falling levelised cost of solar power:


In [ ]:
solar = indicators[indicators["indicator"] == "Solar levelised cost"]

This is trivial with seaborn's `regplot()`:


In [ ]:
sns.regplot(data=solar, x="year", y="value")

plt.show()

Now let's attempt a similar figure with matplotlib:


In [ ]:
import numpy as np

# Calculate the linear relationship
x, y = solar["year"], solar["value"]
lin = np.polyfit(x, y, 1)
pred = np.poly1d(lin)

# Generate the plot
plt.scatter(x, y)
plt.plot(x, pred(x))
plt.xlabel("year")
plt.ylabel("value")

plt.show()

We've used a lot more code, and don't even have a shaded confidence interval!
This highlights a number of drawbacks with solely using matplotlib:

- Matplotlib has no regression functionality, so we have to calculate the linear
  model separately
- The data points and the trend line have to be added via separate function
  calls
- Matplotlib has no functionality for plotting straight from a `DataFrame`
- Because matplotlib cannot access the `DataFrame` labels, we have to supply the
  axis labels manually

### Seaborn doesn't compromise on formatting

If all you want is a quick statistical plot, seaborn is the better choice.
Crucially, because seaborn *wraps around* matplotlib, choosing it doesn't cost
us any of matplotlib's control: we can draw a seaborn plot onto a matplotlib
`Axes` by passing the `ax` argument, then fine-tune with everything we learned
in the previous chapters.


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))

sns.regplot(data=solar, x="year", y="value", ax=ax)

ax.set_xlabel("Year")
ax.set_ylabel("Solar levelised cost / £ per MWh")
ax.set_title("The falling cost of solar power")

plt.show()

So a sensible workflow is:

1. Set a style using matplotlib
2. Initialise the figure using matplotlib's object-oriented interface
3. Insert the desired statistical visualisations using seaborn
4. Customise and fine-tune the figure using matplotlib

## Exercises 1

Please complete *Q1* of [this exercise sheet](3-exercises.ipynb#Q1\))

## 3.2 Facet grids

One place seaborn really shines is **small multiples**: many little plots built
from subsets of a dataset, split on the values of a variable. Seaborn calls
these _facet grids_.

This is exactly how the CCC presents its "key indicators" - one small chart per
indicator. We can reproduce that whole figure with a single `sns.FacetGrid`.

We split into one panel per `indicator` with `col="indicator"`, wrap the panels
over several rows with `col_wrap`, and colour each line by `series`. Because the
indicators are measured in very different units, we let each panel keep its own
$y$-scale with `sharey=False`.

We'll reuse the CCC scenario colours from chapter 2 for the two series - the
historical values and the CCC milestone (Balanced Pathway) projection - so we
import the theme to get its `SCENARIO_COLORS`. The theme turns on constrained
layout, which clashes with the layout a `FacetGrid` manages for itself, so we
switch that off first:


In [ ]:
import textwrap
import jrpyvis.ccc_theme as ccc

plt.rcParams["figure.constrained_layout.use"] = False

palette = {
    "Historical": ccc.SCENARIO_COLORS["Historical"],
    "CCC milestone": ccc.SCENARIO_COLORS["Pathway"],
}

In [ ]:
g = sns.FacetGrid(
    data=indicators,
    col="indicator",
    col_wrap=3,
    hue="series",
    hue_order=["Historical", "CCC milestone"],
    palette=palette,
    sharey=False,
    height=2.3,
    aspect=1.3,
).map(
    sns.lineplot, "year", "value",
).add_legend()

A `FacetGrid` builds its own matplotlib figure internally, so it does not take an
`ax` argument. Instead we reach the underlying figure and axes through the
`.figure` and `.axes` attributes. The indicator names are long, so we shorten each
panel title (dropping the `indicator =` prefix and wrapping it with
`textwrap.fill`), label the $x$-axes, and leave room at the top for an overall
title so it doesn't overlap the panels:


In [ ]:
g = sns.FacetGrid(
    data=indicators,
    col="indicator",
    col_wrap=3,
    hue="series",
    hue_order=["Historical", "CCC milestone"],
    palette=palette,
    sharey=False,
    height=2.3,
    aspect=1.3,
).map(
    sns.lineplot, "year", "value",
).add_legend()

g.set_titles(col_template="{col_name}")
for ax in g.axes.flat:
    ax.set_xlabel("Year")
    ax.set_title(textwrap.fill(ax.get_title(), width=26))

g.figure.subplots_adjust(top=0.90, hspace=0.75)
g.figure.suptitle(
    "Key indicators for the electricity supply sector",
    y=0.985, color=ccc.Colors["vibrant purple"],
)
g.figure.savefig("key_indicators.pdf")

_This recreates the CCC's **Figure 7.5.4**; the original chart and its data are
on sheet `7.5.4` of the workbook._

## Exercises 2

Please complete *Q2* of [this exercise sheet](3-exercises.ipynb#Q2\))

That brings the course to a close - you've gone from matplotlib basics, through
customisation and the CCC house style, to reproducing whole published figures
with seaborn. Happy plotting!
